# Commercialization-channel review

**Status:** graduated companion notebook  
**Research thread:** Award → follow-on contract and firm-level outcomes  
**Canonical computation:** Form D, WS1/WS2, M&A, and capture-recapture scripts under `scripts/data/`

Use this notebook as the template for comparing evidence channels without silently collapsing them into a causal or universal commercialization label.

In [ ]:
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "sbir_etl").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the sbir-analytics checkout")


REPO_ROOT = find_repo_root()
AREA_ID = "nanotechnology"
REPORT_DIR = REPO_ROOT / "data" / "reports" / AREA_ID

## Data contract

The capture matrix is at firm grain. Form D and contract-evidence artifacts may be at award grain. Do not join or compare raw counts until the grains and compound keys are explicit. A channel records observed positive evidence; a false value is not necessarily negative evidence.

In [ ]:
ARTIFACTS = {
    "capture matrix": REPORT_DIR / "capture_recapture.csv",
    "Form D temporal matches": REPORT_DIR / "form_d_post_phase2.csv",
    "contract evidence (WS1)": REPORT_DIR / "ws1_contract_evidence.csv",
    "contract evidence (WS2)": REPORT_DIR / "ws2_contract_evidence.csv",
    "M&A signals": REPORT_DIR / "ma_signal.csv",
}
pd.DataFrame(
    [{"artifact": name, "path": path.relative_to(REPO_ROOT), "exists": path.exists()} for name, path in ARTIFACTS.items()]
)

## Channel overlap

Read the canonical capture matrix rather than recreating its joins here. Near-zero overlap may be structural—for example, when one workstream searches the complement selected by another.

In [ ]:
capture_path = ARTIFACTS["capture matrix"]
if not capture_path.exists():
    print(
        f"Missing {capture_path.relative_to(REPO_ROOT)}. Run "
        f"scripts/data/nano_capture_recapture.py --area {AREA_ID} after its inputs exist."
    )
    capture = pd.DataFrame()
    overlap = pd.DataFrame()
else:
    capture = pd.read_csv(capture_path)
    channels = [column for column in ["fpds", "formd", "strong", "ma", "patent"] if column in capture]
    binary = capture[channels].fillna(0).astype(int)
    overlap = binary.T.dot(binary)
overlap

In [ ]:
if capture.empty:
    capture_history = pd.Series(dtype="int64")
else:
    capture_history = capture["n_channels"].value_counts().sort_index()
capture_history.rename("firms").to_frame()

## Channel-specific diagnostics

Use the generated detail artifacts to inspect lag distributions, confidence tiers, and evidence strength. These diagnostics help challenge a definition; they do not replace the generator or verifier.

In [ ]:
summaries = {}
for name, path in ARTIFACTS.items():
    if path.exists():
        frame = pd.read_csv(path, low_memory=False)
        summaries[name] = {"rows": len(frame), "columns": len(frame.columns)}
pd.DataFrame.from_dict(summaries, orient="index")

## Interpretation log

| Observation | Grain/population | Structural dependence or missingness | Defensible statement |
|---|---|---|---|
| _Draft_ | _Firm or award_ | _State why channels overlap or do not_ | _Descriptive claim only_ |

Before publication, run `nano_verify_report_figures.py` or the area-level verifier and link the resulting findings document.